## Part 1: Setting up the GAN network architecture

In [2]:
import torch
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super().__init__()
        # Noise vector (B, 100) -> fully connected layer to 7*7*128
        self.fc = nn.Linear(z_dim, 7 * 7 * 128)
        self.net = nn.Sequential(
            # 128 x 7 x 7 -> 64 x 14 x 14
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            # 64 x 14 x 14 -> 1 x 28 x 28
            nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),  # output pixels in range [-1, 1]
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 128, 7, 7)   # reshape to (B, 128, 7, 7)
        return self.net(x)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # 1 x 28 x 28 -> 64 x 14 x 14
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            # 64 x 14 x 14 -> 128 x 7 x 7
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),   # single output (logit)
        )

    def forward(self, x):
        # Returns a raw logit; apply torch.sigmoid() to get the real/fake probability.
        # During training we use BCEWithLogitsLoss, which applies sigmoid internally.
        return self.net(x)

In [3]:
if __name__ == "__main__":
    G, D = Generator(), Discriminator()
    z = torch.randn(16, 100)
    fake = G(z)
    print(fake.shape)      # torch.Size([16, 1, 28, 28])
    print(D(fake).shape)   # torch.Size([16, 1])

torch.Size([16, 1, 28, 28])
torch.Size([16, 1])


## Part2 Use MNIST dataset to train GAN model

In [4]:
import os
os.makedirs("app", exist_ok=True)
open("app/__init__.py", "w").close()

In [5]:
%%writefile app/main.py
import io
from pathlib import Path

import torch
from fastapi import FastAPI, Query
from fastapi.responses import StreamingResponse
from torchvision.utils import save_image

from helper_lib.model import get_model
from helper_lib.utils import load_model

app = FastAPI(title="GAN Digit Generator API")

# ---- Load the trained GAN once at startup ----
MODEL_PATH = Path(__file__).resolve().parent.parent / "gan_mnist.pth"
gan_model = load_model(get_model("GAN"), MODEL_PATH, device="cpu")


@app.get("/")
def root():
    """Health check endpoint."""
    return {"message": "GAN API is running. Go to /docs to try it."}


@app.get("/generate_digit")
def generate_digit(num_images: int = Query(16, ge=1, le=64)):
    """Generate handwritten digits with the GAN and return them as a PNG grid."""
    with torch.no_grad():
        z = torch.randn(num_images, gan_model.z_dim)       # random latent vectors
        imgs = gan_model.generator(z) * 0.5 + 0.5          # rescale [-1, 1] -> [0, 1]
    buf = io.BytesIO()
    save_image(imgs, buf, format="png", nrow=8)            # write image grid to memory
    buf.seek(0)
    return StreamingResponse(buf, media_type="image/png")

Overwriting app/main.py


In [7]:
with open("requirements.txt", "w") as f:
    f.write("torch\ntorchvision\nfastapi\nuvicorn\nmatplotlib\nnumpy\n")

with open(".gitignore", "w") as f:
    f.write("data/\n__pycache__/\n*.pyc\n.ipynb_checkpoints/\n.DS_Store\n")

In [8]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

# Install CPU-only PyTorch first (much smaller image), then the other dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir torch torchvision --index-url https://download.pytorch.org/whl/cpu \
    && pip install --no-cache-dir -r requirements.txt

# Copy the helper library, API code, and trained model weights
COPY helper_lib ./helper_lib
COPY app ./app
COPY gan_mnist.pth .

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

Overwriting Dockerfile


In [9]:
%%writefile .dockerignore
data/
.git/
__pycache__/
*.pyc
.ipynb_checkpoints/
*.ipynb
.DS_Store

Overwriting .dockerignore
